# Notebook 3 — Comprensione del testo en dati visuales

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lezione 3:** Understanding Text in Visual Data (rilevamento de testo en immagini, escritura manuscrita y file PDF/TIFF)

En este notebook practicarás las tres variprima di reconocimiento de testo (OCR, reconocimiento óptico de caracteres) que ofrece la Vision API.

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## 1. Detectar testo en una immagine (`text_detection`)

El metodo `text_detection` localiza regiones con testo y reconoce sus caracteres, impresos o manuscritos, en varios idiomas. Junto al testo te devuelve el idioma rilevato (`locale`), una punteggio de confidenza y el riquadro delimitador (`bounding_poly`) de cada fragmento.

El primer elemento de `response.text_annotations` contiene **todo** el testo rilevato en la immagine; los siguientes elementos son cada palabra o fragmento por separado, con su propio `bounding_poly`.

In [ ]:
def detect_text(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.text_detection(image=image)
    annotations = response.text_annotations

    if annotations:
        return annotations[0].description
    return None


# texto = detect_text("cartel.jpg")
# print(texto)


### Variante: testo en una immagine alojada en una URL

En lugar de leer bytes de un file local, puedes asignar la URL directamente a `image.source.image_uri`.

In [ ]:
def detect_text_from_url(image_url):
    image = vision.Image()
    image.source.image_uri = image_url

    response = client.text_detection(image=image)
    annotations = response.text_annotations

    if annotations:
        return annotations[0].description
    return None


## Ejercicio 1 — Palabra por palabra con sus coordenadas

Completa `rilevare_palabras` para que, dada una immagine, devuelva una lista de diccionarios con la palabra (`text`) y las coordenadas de su `bounding_poly` (`vertices`), usando **todos** los elementos de `text_annotations` excepto el primero (que es el testo completo).

In [ ]:
def detectar_palabras(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.text_detection(image=image)

    # TODO: recorre response.text_annotations[1:] y construye la lista de diccionarios
    # cada diccionario debe tener las claves "text" y "vertices"
    palabras = ...
    return palabras


# print(detectar_palabras("cartel.jpg"))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def rilevare_palabras(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.text_detection(image=image)

    palabras = []
    for annotation in response.text_annotations[1:]:
        vertices = [(v.x, v.y) for v in annotation.bounding_poly.vertices]
        palabras.append({"text": annotation.description, "vertices": vertices})

    return palabras


print(rilevare_palabras("cartel.jpg"))
```

</details>

## 2. Limpiar el testo extraído con pandas

Una vez que tienes el testo en un DataFrame, puedes normalizarlo con expresiones regulares y con los metodi de cadena de pandas.

In [ ]:
import pandas as pd
import re

df = pd.DataFrame({"text": ["  ¡Hola, MUNDO!  ", "Precio: 25€ (oferta)"]})

df["text_limpio"] = df["text"].apply(lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", x))
df["text_limpio"] = df["text_limpio"].str.strip().str.lower()
df


## 3. Testo manuscrito y documentos (`document_text_detection`)

El testo manuscrito per di più difícil de reconocer que el impreso: la escritura varía mucho de una persona a otra. Para documentos (currículums, notas manuscritas, formularios), usa **`document_text_detection`** en lugar de `text_detection`. El risultato se organiza jerárquicamente: **páginas → bloques → párrafos → palabras → símbolos**, y cada livello trae su propio `confidence` (valor entre cero y uno).

In [ ]:
def extract_text_from_document(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.document_text_detection(image=image)
    document = response.full_text_annotation

    for page in document.pages:
        for block in page.blocks:
            print("Confianza del bloque:", block.confidence)
            for paragraph in block.paragraphs:
                print("  Confianza del párrafo:", paragraph.confidence)
                for word in paragraph.words:
                    word_text = "".join([symbol.text for symbol in word.symbols])
                    print("   ", word_text)

    return document.text


# texto_completo = extract_text_from_document("nota_manuscrita.jpg")
# print(texto_completo)


## Ejercicio 2 — Comparar `text_detection` frente a `document_text_detection`

Completa `comparar_metodos` para que, dada la misma immagine, devuelva un diccionario con:

- `"testo_simple"`: el risultato de `text_detection` (primer elemento de `text_annotations`)
- `"testo_documento"`: el risultato de `document_text_detection` (`full_text_annotation.text`)
- `"confidenza_media_bloques"`: la media de `confidence` de todos los bloques del documento

Pista: usa la funzione `extract_text_from_document` de arriba come referencia para recorrer los bloques.

In [ ]:
def comparar_metodos(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()
    image = vision.Image(content=content)

    # TODO 1: llama a text_detection y guarda el texto completo
    texto_simple = ...

    # TODO 2: llama a document_text_detection
    response_doc = ...
    document = response_doc.full_text_annotation

    # TODO 3: recorre los bloques de document.pages y calcula la confianza media
    confianzas = []
    for page in document.pages:
        for block in page.blocks:
            ...

    confianza_media = sum(confianzas) / len(confianzas) if confianzas else 0

    return {
        "texto_simple": texto_simple,
        "texto_documento": document.text,
        "confianza_media_bloques": confianza_media,
    }


# print(comparar_metodos("documento.jpg"))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def comparar_metodos(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()
    image = vision.Image(content=content)

    response_simple = client.text_detection(image=image)
    testo_simple = response_simple.text_annotations[0].description if response_simple.text_annotations else ""

    response_doc = client.document_text_detection(image=image)
    document = response_doc.full_text_annotation

    confidenzas = []
    for page in document.pages:
        for block in page.blocks:
            confidenzas.append(block.confidence)

    confidenza_media = sum(confidenzas) / len(confidenzas) if confidenzas else 0

    return {
        "testo_simple": testo_simple,
        "testo_documento": document.text,
        "confidenza_media_bloques": confidenza_media,
    }


print(comparar_metodos("documento.jpg"))
```

</details>

## 4. Testo en file PDF y TIFF

Para procesar file completos —no solo immagini sueltas— en formato PDF o TIFF (formato de file de immagine etichettado), combina la Vision API con **Google Cloud Storage (GCS)**, el servicio de almacenamiento en la nube de Google. El file debe residir en un **bucket** de GCS prima di que la API pueda procesarlo.

**Límite y precio orientativo**: cada file admite un máximo de dos mil páginas; el precio se calcula por unidad (una página = una unidad), con mil unidades gratuitas al mes.

```bash
pip install --upgrade google-cloud-storage
```

**Flujo asíncrono recomendado** (procesamiento por lotes vía GCS):

In [ ]:
from google.cloud import vision

def procesar_pdf_en_gcs(gcs_source_uri, gcs_destination_uri):
    feature = vision.Feature(type_=vision.Feature.Type.DOCUMENT_TEXT_DETECTION)

    gcs_source = vision.GcsSource(uri=gcs_source_uri)
    input_config = vision.InputConfig(gcs_source=gcs_source, mime_type="application/pdf")

    gcs_destination = vision.GcsDestination(uri=gcs_destination_uri)
    output_config = vision.OutputConfig(gcs_destination=gcs_destination, batch_size=2)

    async_request = vision.AsyncAnnotateFileRequest(
        features=[feature], input_config=input_config, output_config=output_config
    )

    operation = client.async_batch_annotate_files(requests=[async_request])
    print("Procesando...")
    operation.result(timeout=180)
    print("Listo. Revisa el bucket de salida:", gcs_destination_uri)


# procesar_pdf_en_gcs("gs://tu-bucket/entrada.pdf", "gs://tu-bucket/salida/")


In [ ]:
from google.cloud import storage
import re
import json

def leer_resultado_gcs(gcs_destination_uri):
    storage_client = storage.Client()

    match = re.match(r"gs://([^/]+)/(.+)", gcs_destination_uri)
    bucket_name = match.group(1)
    prefix = match.group(2)

    bucket = storage_client.get_bucket(bucket_name)
    blob_list = list(bucket.list_blobs(prefix=prefix))

    output = blob_list[0]
    json_string = output.download_as_string()
    response_json = json.loads(json_string)

    return response_json["responses"][0]["fullTextAnnotation"]["text"]


# print(leer_resultado_gcs("gs://tu-bucket/salida/"))


### Alternativa sin GCS: convertir el PDF a immagini localmente

Si prefieres no usar un bucket para practicar, convierte el PDF a immagini con `pdf2image` y aplica `text_detection` página a página (más sencillo de montar, anche se pierdes las ventajas del procesamiento asíncrono por lotes).

In [ ]:
import io
from pdf2image import convert_from_bytes

def detectar_texto_pdf_local(pdf_path):
    with open(pdf_path, "rb") as f:
        content = f.read()

    images = convert_from_bytes(content)
    textos_por_pagina = []

    for image in images:
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG")
        response = client.text_detection(image=vision.Image(content=buffer.getvalue()))
        if response.text_annotations:
            textos_por_pagina.append(response.text_annotations[0].description)
        else:
            textos_por_pagina.append("")

    return textos_por_pagina


# paginas = detectar_texto_pdf_local("contrato.pdf")
# for i, texto in enumerate(paginas):
#     print(f"--- Página {i + 1} ---")
#     print(texto)


## Ejercicio 3 — Calcula el coste estimado de un lote de documentos

Completa `costo_estimado` para que, dado un diccionario `{"nombre_file": num_paginas}`, calcule el coste total aplicando la regla "una página = una unidad" y un precio de 0,60 $ por unidad, y devuelva tanto el coste total come un desglose por file.

In [ ]:
PRECIO_POR_UNIDAD = 0.60

def costo_estimado(paginas_por_archivo):
    # TODO: calcula el desglose {"archivo": costo} y el total
    desglose = ...
    total = ...
    return desglose, total


# desglose, total = costo_estimado({"contrato_a.pdf": 12, "contrato_b.pdf": 8})
# print(desglose, total)


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
PRECIO_POR_UNIDAD = 0.60

def costo_estimado(paginas_por_file):
    desglose = {nombre: paginas * PRECIO_POR_UNIDAD for nombre, paginas in paginas_por_file.items()}
    total = sum(desglose.values())
    return desglose, total


desglose, total = costo_estimado({"contrato_a.pdf": 12, "contrato_b.pdf": 8})
print(desglose, total)
```

</details>

---

**Siguiente paso:** abre `04_Comprension_de_Immagini.ipynb` para trabajar con crop hints, rilevamento de volti y propiedades de la immagine.